In [1]:
%pip install anthropic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [35]:
import json

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"


In [31]:
from dotenv import load_dotenv
from anthropic import Anthropic

# Lee el archivo .env de tu explorador de forma automática
load_dotenv()

True

In [32]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [36]:
def generate_dataset():
    prompt = """
            Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

            Example output:
            ```json
            [
            {
                "task": "Description of task",
            },
            ...additional
            ]
            ```

            * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
            * Focus on tasks that do not require writing much code

            Please generate 3 objects.
            """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")


    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [39]:
# Llama a la API (prefill + stop) y parsea la respuesta a lista/dict de Python
dataset = generate_dataset()
print(dataset)

# with open: abre el archivo, escribe y lo cierra solo al salir del bloque (aunque haya error)
# "w" = write (crea o SOBRESCRIBE dataset.json en la carpeta del notebook)
# json.dump convierte la lista/dict de Python a texto JSON en el archivo
# indent=2 = sangría legible (cada nivel con 2 espacios)
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with keys 'bucket' and 'key'"}, {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to all objects in an S3 bucket named 'my-data-bucket'"}, {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters)'}]


In [40]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
            Please solve the following task:

            {test_case["task"]}
            """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output
    

In [43]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""

    output = run_prompt(test_case)

    # TODO: Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [44]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""

    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [45]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [47]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a Python function that parses AWS S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri):\n    \"\"\"\n    Parse an AWS S3 bucket URI and return bucket name and key.\n    \n    Args:\n        uri (str): S3 URI in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' keys\n        \n    Raises:\n        ValueError: If the URI format is invalid\n        \n    Examples:\n        >>> parse_s3_uri('s3://my-bucket/path/to/file.txt')\n        {'bucket': 'my-bucket', 'key': 'path/to/file.txt'}\n        \n        >>> parse_s3_uri('s3://my-bucket/')\n        {'bucket': 'my-bucket', 'key': ''}\n    \"\"\"\n    if not isinstance(uri, str):\n        raise ValueError(\"URI must be a string\")\n    \n    if not uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_scheme = uri[5:]\n    \n    # Split on the 